# Sparse GCN Benchmark: Keras vs TensorFlow

This notebook benchmarks representative sparse GCN implementations that use the same graph kernel pattern:
- sparse adjacency times dense node matrix (`tf.sparse.sparse_dense_matmul`)
- optional JIT/XLA compilation (`tf.function(jit_compile=True)`)

Goal: measure whether backend-agnostic Keras abstractions can stay close to pure TensorFlow performance without custom C++/CUDA kernels.

## Backend notes

- In TensorFlow backend mode, both implementations ultimately dispatch to TensorFlow sparse kernels.
- The main performance differences come from Python overhead, graph tracing boundaries, and fusion opportunities.
- `jit_compile=True` can reduce launch overhead and fuse surrounding dense ops, but improvements depend on matrix sizes and hardware.

In [ ]:
import time
import numpy as np
import tensorflow as tf
import keras

from tf_gnns.models.gcn import SparseGCNConv

print('TF version:', tf.__version__)
print('Keras backend:', keras.backend.backend())

In [ ]:
def make_random_graph(num_nodes, avg_degree, seed=0):
    rng = np.random.default_rng(seed)
    num_edges = num_nodes * avg_degree
    senders = rng.integers(0, num_nodes, size=num_edges, dtype=np.int32)
    receivers = rng.integers(0, num_nodes, size=num_edges, dtype=np.int32)
    edge_weights = rng.uniform(0.5, 1.5, size=num_edges).astype(np.float32)

    diag = np.arange(num_nodes, dtype=np.int32)
    senders = np.concatenate([senders, diag])
    receivers = np.concatenate([receivers, diag])
    edge_weights = np.concatenate([edge_weights, np.ones((num_nodes,), dtype=np.float32)])

    row = receivers
    col = senders
    deg = np.zeros((num_nodes,), dtype=np.float32)
    np.add.at(deg, row, edge_weights)
    deg = np.maximum(deg, 1e-12)
    inv_sqrt = 1.0 / np.sqrt(deg)
    norm_w = edge_weights * inv_sqrt[row] * inv_sqrt[col]

    idx = np.stack([row, col], axis=1)
    sp_adj = tf.sparse.reorder(tf.sparse.SparseTensor(
        indices=tf.constant(idx, dtype=tf.int64),
        values=tf.constant(norm_w, dtype=tf.float32),
        dense_shape=tf.constant([num_nodes, num_nodes], dtype=tf.int64),
    ))
    return sp_adj

def make_inputs(num_nodes, feat_dim, seed=0):
    rng = np.random.default_rng(seed)
    x = tf.constant(rng.normal(size=(num_nodes, feat_dim)).astype(np.float32))
    return x

In [ ]:
@tf.function
def pure_tf_gcn_step(sp_adj, x, kernel):
    h = tf.sparse.sparse_dense_matmul(sp_adj, x)
    return tf.linalg.matmul(h, kernel)

@tf.function(jit_compile=True)
def pure_tf_gcn_step_jit(sp_adj, x, kernel):
    h = tf.sparse.sparse_dense_matmul(sp_adj, x)
    return tf.linalg.matmul(h, kernel)

In [ ]:
def make_td_from_sparse(sp_adj, x):
    idx = tf.cast(sp_adj.indices, tf.int32)
    row = idx[:, 0]
    col = idx[:, 1]
    edge_w = sp_adj.values
    num_nodes = int(x.shape[0])
    num_edges = int(edge_w.shape[0])

    td = {
        'nodes': x,
        'edges': tf.zeros((num_edges, 1), dtype=tf.float32),
        'senders': col,
        'receivers': row,
        'n_edges': tf.constant([num_edges], dtype=tf.int32),
        'n_nodes': tf.constant([num_nodes], dtype=tf.int32),
        'n_graphs': tf.constant(1, dtype=tf.int32),
        'global_attr': None,
        'global_reps_for_edges': tf.zeros((num_edges,), dtype=tf.int32),
        'global_reps_for_nodes': tf.zeros((num_nodes,), dtype=tf.int32),
        'edge_weights': edge_w,
    }
    return td

In [ ]:
def benchmark(fn, warmup=20, iters=100):
    for _ in range(warmup):
        _ = fn()
    start = time.perf_counter()
    for _ in range(iters):
        _ = fn()
    end = time.perf_counter()
    return (end - start) / iters * 1e3

def run_case(num_nodes, feat_in=64, feat_out=64, avg_degree=16, seed=0):
    sp_adj = make_random_graph(num_nodes, avg_degree, seed=seed)
    x = make_inputs(num_nodes, feat_in, seed=seed + 1)
    kernel = tf.Variable(tf.random.normal((feat_in, feat_out), stddev=0.1))

    td = make_td_from_sparse(sp_adj, x)
    keras_layer = SparseGCNConv(feat_out, activation=None, add_self_loops=False, normalize=False, jit_compile=False)
    keras_layer_jit = SparseGCNConv(feat_out, activation=None, add_self_loops=False, normalize=False, jit_compile=True)

    _ = keras_layer(td)
    _ = keras_layer_jit(td)

    keras_layer.kernel.assign(kernel)
    keras_layer_jit.kernel.assign(kernel)

    ms_tf = benchmark(lambda: pure_tf_gcn_step(sp_adj, x, kernel))
    ms_tf_jit = benchmark(lambda: pure_tf_gcn_step_jit(sp_adj, x, kernel))
    ms_keras = benchmark(lambda: keras_layer(td)['nodes'])
    ms_keras_jit = benchmark(lambda: keras_layer_jit(td)['nodes'])

    return {
        'nodes': num_nodes,
        'edges': int(sp_adj.values.shape[0]),
        'tf_ms': ms_tf,
        'tf_jit_ms': ms_tf_jit,
        'keras_ms': ms_keras,
        'keras_jit_ms': ms_keras_jit,
    }

In [ ]:
cases = [2000, 5000, 10000]
results = [run_case(n) for n in cases]
results

## Interpreting results

Typical observations:
- Pure TensorFlow may be slightly faster for tiny graphs due to lower framework overhead.
- At moderate and large graph sizes, kernel time dominates and Keras/TensorFlow often converge.
- `jit_compile=True` can improve throughput if your runtime/hardware supports efficient XLA for sparse+dense patterns.

Backend behavior notes:
- The key kernel is sparse-dense SpMM. This is where most time goes for sparse GCN.
- Dense projection (`matmul`) is often fused or at least efficiently scheduled around SpMM.
- End-to-end speed is sensitive to index sorting, tensor shapes, and retracing frequency.

## Recommendations for this library

1. Keep a backend-agnostic public API while adding backend-specific fast paths internally.
2. Prefer sparse adjacency paths over edge-wise gather/segment for large sparse graphs.
3. Add microbenchmarks to CI (smoke-size) and periodic large benchmarks for regressions.
4. Cache preprocessed graph structures (sorted sparse indices, normalized weights) when topology is static.
5. Keep pure Python loops out of hot paths; rely on compiled TensorFlow graph functions.